# Similar Movies Debug Runner

Run the setup cell once, set `tmdb_ids`, then execute the result cell to inspect the standalone similar-movies flow with lane labels.

## Cell 1 - Setup

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from db.postgres import fetch_movie_cards, pool as postgres_pool
from search_v2.similar_movies import run_similar_movies_for_ids

if postgres_pool._closed:
    await postgres_pool.open()

print(f"Project root: {PROJECT_ROOT}")
print("Postgres pool open")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/michaelkeohane/Documents/movie-finder-rag
Postgres pool open


## Cell 2 - Anchors

## Cell 3 - Results

In [2]:
tmdb_ids = [597, 87827]
limit = 25

def year_from_release_ts(release_ts: int | None) -> int | None:
    if release_ts is None:
        return None
    return datetime.fromtimestamp(release_ts, tz=timezone.utc).year


result = await run_similar_movies_for_ids(tmdb_ids, limit=limit)

cards = await fetch_movie_cards([item.movie_id for item in result.ranked])
cards_by_id = {card["movie_id"]: card for card in cards}

rows = []
for rank, item in enumerate(result.ranked, start=1):
    card = cards_by_id.get(item.movie_id, {})
    lane_scores = item.evidence.lane_scores
    rows.append(
        {
            "rank": rank,
            "title": card.get("title", "<missing card>"),
            "year": year_from_release_ts(card.get("release_ts")),
            "tmdb_id": item.movie_id,
            "score": round(item.score, 4),
            "dominant_lane": item.evidence.dominant_lane,
            "lanes": ", ".join(item.evidence.candidate_sources),
            "shape": round(lane_scores.get("shape", 0.0), 4),
            "director": round(lane_scores.get("director", 0.0), 4),
            "franchise": round(lane_scores.get("franchise", 0.0), 4),
            "studio": round(lane_scores.get("studio", 0.0), 4),
            "source": round(lane_scores.get("source", 0.0), 4),
            "quality": round(lane_scores.get("quality", 0.0), 4),
        }
    )

print("anchors:", result.anchor_movie_ids)
print("active_anchor_types:", result.active_anchor_types)
print("lane_weights:", result.debug.normalized_lane_weights)
print("vector_space_weights:", result.debug.vector_space_weights)
display(pd.DataFrame(rows))

anchors: [597, 87827]
active_anchor_types: ['standard_shape']
lane_weights: {'shape': 0.6777419646009603, 'franchise': 0.0, 'source': 0.0, 'quality': 0.0, 'format': 0.06445160707980795, 'themes': 0.19335482123942382, 'cast': 0.0, 'specific_award': 0.06445160707980795, 'director': 1.0, 'studio': 0.0}
vector_space_weights: {'anchor': 0.10058243469967648, 'plot_events': 0.041589491359864644, 'plot_analysis': 0.08317898271972929, 'viewer_experience': 0.2559353314453209, 'watch_context': 0.15105687752846841, 'narrative_techniques': 0.1627334083391915, 'production': 0.06078464121826371, 'reception': 0.14413883268948505}


,rank,title,year,tmdb_id,score,dominant_lane,lanes,shape,director,franchise,studio,source,quality
0,1,Titanic,1953,16535,0.8463,shape,"shape, format, themes, specific_award",1.0000,0.0,0.0,0.0,0.0,0.0
1,2,The Book Thief,2013,203833,0.8152,shape,"shape, format, themes, specific_award",0.9596,0.0,0.0,0.0,0.0,0.0
2,3,Adrift,2018,429300,0.7843,shape,"shape, format, themes",0.9286,0.0,0.0,0.0,0.0,0.0
3,4,Cast Away,2000,8358,0.7573,shape,"shape, format, themes, specific_award",0.7157,0.0,0.0,0.0,0.0,0.0
4,5,What Dreams May Come,1998,12159,0.7355,shape,"shape, format, themes, specific_award",0.7698,0.0,0.0,0.0,0.0,0.0
5,6,Gravity,2013,49047,0.6631,shape,"shape, format, themes, specific_award",0.4387,0.0,0.0,0.0,0.0,0.0
6,7,Christmas in Conway,2013,240820,0.6351,shape,"shape, format, themes",0.7190,0.0,0.0,0.0,0.0,0.0
7,8,"Nestor, the Long-Eared Christmas Donkey",1977,26537,0.6255,shape,"shape, format, themes",0.7055,0.0,0.0,0.0,0.0,0.0
8,9,Lolo and the Kid,2024,1317887,0.6118,shape,"shape, format, themes",0.7310,0.0,0.0,0.0,0.0,0.0
9,10,The Curious Case of Benjamin Button,2008,4922,0.6093,shape,"shape, format, themes, specific_award",0.3369,0.0,0.0,0.0,0.0,0.0
